In [ ]:
# baixando o dataset do tiny shakespeare
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-06 16:05:26--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-07-06 16:05:26 (21.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# lendo o dataset
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
print('Tamanho do dataset em char: ', len(text))

Tamanho do dataset em char:  1115394


In [ ]:
# vendo os 1000 primeiros char
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# caracteres unicos que aparecem no texto
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# mapeando os caracteres para int
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
# encoder pega uma string e transforma em uma lista de inteiros
encode = lambda s: [stoi[c] for c in s]
# decoder pega a lista de interes e transforma na string
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# fazer o encode em todo o dataset
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
# dividindo os dados entre treino e validação
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
# block size é o contexto maximo para predição
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
# mostrando um exemplo de previsão, onde mostramos qual a mensagem esperada (target) dependendo do contexto
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f'quando o contexto é {context} o target é {target}')

quando o contexto é tensor([18]) o target é 47
quando o contexto é tensor([18, 47]) o target é 56
quando o contexto é tensor([18, 47, 56]) o target é 57
quando o contexto é tensor([18, 47, 56, 57]) o target é 58
quando o contexto é tensor([18, 47, 56, 57, 58]) o target é 1
quando o contexto é tensor([18, 47, 56, 57, 58,  1]) o target é 15
quando o contexto é tensor([18, 47, 56, 57, 58,  1, 15]) o target é 47
quando o contexto é tensor([18, 47, 56, 57, 58,  1, 15, 47]) o target é 58


In [ ]:
torch.manual_seed(1337)
# quantidade de sequencias a serem processadas paralelamente
batch_size = 4
# maximo de contexto para as previsões
block_size = 8

def get_batch(split):
  # gera um lote de dados com inputs e targets
  # separa entre treino e validacao
  data = train_data if split == 'train' else val_data
  # gera inteiros aleatórios
  ix = torch.randint(len(data) - block_size, (batch_size,))
  # separa entre input e target
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

# pegando os valores aleatórios
xb, yb = get_batch('train')

# mostrando os valores
print('Inputs:')
print(xb.shape)
print(xb)
print('Targets:')
print(yb.shape)
print(yb)

print('----')

# para os valores das sequencias
for b in range(batch_size):
  # para a quantidade de contexto máximo
  for t in range(block_size):
    # mostra o contexto e o target
    context = xb[b, :t+1]
    target = yb[b, t]
    print(f'quando o contexto é {context.tolist()} o target é {target}')

Inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
quando o contexto é [24] o target é 43
quando o contexto é [24, 43] o target é 58
quando o contexto é [24, 43, 58] o target é 5
quando o contexto é [24, 43, 58, 5] o target é 57
quando o contexto é [24, 43, 58, 5, 57] o target é 1
quando o contexto é [24, 43, 58, 5, 57, 1] o target é 46
quando o contexto é [24, 43, 58, 5, 57, 1, 46] o target é 43
quando o contexto é [24, 43, 58, 5, 57, 1, 46, 43] o target é 39
quando o contexto é [44] o target é 53
quando o contexto é [44, 53] o target é 56
quando o contexto é [44, 53, 56] o target é 1
quando o contexto é [44, 53, 56, 1] o target é 58
quando o c

In [ ]:
# output do transformer
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# modelo
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # lookup table que mapeia cada token para um vetor.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # pega os logits da tabela
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            # pega as dimensões do tensor de logits
            B, T, C = logits.shape

            # achatando as dimensões em uma só
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            # calcula a loss do modelo
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # pega os logits do contexto atual
            logits, loss = self(idx)

            # pega o caractere mais recente da sequência
            # transforma o formato de (B, T, C) para (B, C)
            logits = logits[:, -1, :]

            # aplica a função softmax
            probs = F.softmax(logits, dim=-1)

            # prevê o próximo token
            idx_next = torch.multinomial(probs, num_samples=1)

            # Concatena o novo token
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# instancia o modelo
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# gerando um texto para teste
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
# criando um optimizador
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
# treinando o modelo
batch_size = 32

for steps in range(10000):

  xb, yb = get_batch('train')

  # calcula e avalia a loss
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.3132691383361816


In [ ]:
# gerando um texto para teste
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=400)[0].tolist()))


Whathe f, que, t-bldu aino athe tef horw; h? ere whead thacedefetht panken thon. d bokit tese;

If d, n blirstard,
Al?
BENG IZANue uistw t tonthoney pr anent BRKIthaghatens:
Lofom, at yo myond RIS:
Ledin rond, FIN!
Myod ohey, n icu d!--etithin rreanne sthacanoxcises hed ngh' o shee'trt.

OSTow wo ce hefolat ctecicrsellliny;

Tor woucthend ble wim, in tofie,
FIOffo;
Foknkerod, s st:
Bu, an aves!
LI


In [ ]:
# multiplicação de matriz mostrando uma possivel abordagem para carregar
# contexto dos tokens anteriores
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# olhando o exemplo
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# passando a média dos vetores anteriores como contexto para o próximo
# através de um loop
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [ ]:
# fazendo a mesma coisa, mas agora utilizando multplicação de matrizes
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [ ]:
# mesma coisa, mas agora utilizando a função softmax para atribuir
# pesos reais aos tokens que importam
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# projetando os valores em um espaço de 16 dimensões
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
# calculando a afinidade
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

# mascarando e normalizando
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

out.shape

torch.Size([4, 8, 16])

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [46]:
k.var()

tensor(1.0449)

In [47]:
q.var()

tensor(1.0700)

In [48]:
wei.var()

tensor(1.0918)

In [49]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [50]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [51]:
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps

    # parametros aprendiveis
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calcula a média das features (ignora o tamanho do batch)
    xmean = x.mean(1, keepdim=True)
    xvar = x.var(1, keepdim=True)

    # normalização
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps)

    # aplica a transformação com os parametros
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    # retorna os parametros para o otimizador atualizar
    return [self.gamma, self.beta]

# testando a camada
torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])